<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>

# Optimizing LLM as a Judge Prompts

**What is LLM as a Judge?**

An LLM as a Judge refers to using an LLM as a tool for evaluating and scoring responses based on predefined criteria. The LLM functions as a "judge," making decisions and assigning scores based on the input it receives. Some common use cases include evaluations for hallucinations, user frustration, and summarization quality.

**Why Do We Want to Refine These Prompts?**

While LLMs are powerful tools for evaluation, their performance can be inconsistent. Factors like ambiguity in the prompt, biases in the model, or a lack of clear guidelines can lead to unreliable results. By fine-tuning your LLM prompts, you can improve the model's consistency, fairness, and accuracy, ensuring it delivers more reliable evaluations.

In this tutorial, you will:
- Generate an LLM as a Judge evaluation prompt and test it against a dataset
- Learn about various optimization techniques to improve the template, measuring accuracy at each step
- Understand how to apply these techniques together for better evaluation across your specific use cases

You'll need an **OpenAI API key** for this tutorial.

Let's get started!

# Set Up Dependencies and Keys

In [ ]:
import sys
!{sys.executable} -m pip install -q openai datasets pandas tqdm nest-asyncio

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


# Load Dataset

In this tutorial, we will focus on creating an LLM as a Judge prompt designed to assess **empathy and emotional intelligence** in chatbot responses. This is especially useful for use cases like mental health chatbots or customer support interactions.

We will start by loading a dataset containing 30 chatbot responses, each with a ground-truth empathy score (out of 10). Throughout the tutorial, we'll use our prompt to evaluate these responses and compare the output to the ground-truth labels — this lets us measure how well each prompt variant performs.

In [ ]:
import uuid
import pandas as pd
from datasets import load_dataset

ds = load_dataset("syeddula/empathy_scores")["test"]
dataset = ds.to_pandas()
unique_id = uuid.uuid4()

print(f"Dataset loaded: {len(dataset)} rows  (id: empathy-{unique_id})")
dataset.head()

Dataset loaded: 30 rows  (id: empathy-54f97eaa-2a4c-4fd8-970c-eb7c3271c48b)


,AI_Response,EI_Empathy_Score
0,You’ll get over it. Just try not to think too ...,1
1,This happens to everyone. It’s not a big deal.,1
2,I can’t really help with that. Maybe talk to s...,1
3,"That sounds difficult. Anyway, what else can I...",2
4,I guess that’s frustrating. Let me know if you...,2


# Core Evaluation Helpers

Before running any experiments, we define two reusable helpers that will be used throughout the notebook:

- **`llm_score`**: Sends a response + evaluation template to GPT-4 and parses back a 1–10 integer score.
- **`run_experiment`**: Iterates over the dataset, calls a task function on each row, passes the result to an evaluator, and reports accuracy.
- **`evaluate_response`**: Compares the predicted score to the ground-truth label — a prediction is considered correct if it is within ±2 of the true score.

In [ ]:
import re
import pandas as pd
import nest_asyncio
from tqdm import tqdm
from openai import OpenAI

nest_asyncio.apply()
client = OpenAI()


def llm_score(ai_response, template, model="gpt-4"):
    """Send a response to GPT-4 for scoring using the given evaluation template."""
    prompt = (
        template
        + f'\n\n---\nAI Response to evaluate:\n"{ai_response}"'
        + "\n\nRespond with ONLY a single integer score from 1 to 10:"
    )
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=10,
        )
        raw = response.choices[0].message.content.strip()
        match = re.search(r'\b(10|[1-9])\b', raw)
        return int(match.group(1)) if match else None
    except Exception as e:
        print(f"Scoring error: {e}")
        return None


def run_experiment(dataset_df, task, evaluators, experiment_name):
    """Run a task over the full dataset and report evaluator accuracy."""
    results = []
    print(f"\n▶  Experiment: '{experiment_name}'  ({len(dataset_df)} samples)")
    for _, row in tqdm(dataset_df.iterrows(), total=len(dataset_df)):
        inp = row.to_dict()
        output = task(inp)
        evals = {fn.__name__: fn(inp, output) for fn in evaluators}
        results.append({"output": output, **evals})

    df = pd.DataFrame(results)
    for fn in evaluators:
        acc = df[fn.__name__].mean()
        print(f"   {fn.__name__}: {acc:.1%}  ({int(df[fn.__name__].sum())}/{len(df)})")
    return df


def evaluate_response(inp, output):
    """A prediction is correct if it is within ±2 of the ground-truth score."""
    expected_score = inp["EI_Empathy_Score"]
    predicted_score = output if output is not None else 5
    return abs(expected_score - predicted_score) <= 2


print("Helpers defined.")

Helpers defined.


# Generate LLM as a Judge Template using Meta Prompting

Before iterating on our template, we need to establish a baseline prompt. Running the cell below will generate an LLM as a Judge prompt specifically for evaluating empathy and emotional intelligence using **meta prompting** — asking GPT-4 to write the evaluation rubric for us.

When generating this template, we emphasize:
- Picking evaluation criteria (e.g., empathy, emotional support, emotional intelligence)
- Defining a clear scoring system (1–10 scale with defined descriptions)
- Setting response formatting guidelines for clarity and consistency
- Including an explanation for why the LLM selects a given score

In [ ]:
def generate_eval_template():
    meta_prompt = """
    You are an expert in AI evaluation and emotional intelligence assessment. Your task is to create a structured evaluation template for assessing the emotional intelligence and empathy of AI responses to user inputs.

    ### Task Overview:
    Generate a detailed evaluation template that measures the AI's ability to recognize user emotions, respond empathetically, and provide emotionally appropriate responses. The template should:
    - Include 3 to 5 distinct evaluation criteria that assess different aspects of emotional intelligence.
    - Define a scoring system on a scale of 1 to 10, ensuring a broad distribution of scores across different responses.
    - Provide clear, tiered guidelines for assigning scores, distinguishing weak, average, and strong performance.
    - Include a justification section requiring evaluators to explain the assigned score with specific examples.
    - Ensure the scoring rubric considers complexity and edge cases, preventing generic or uniform scores.

    ### Format:
    Return the evaluation template as plain text, structured with headings, criteria, and a detailed scoring rubric. The template should be easy to follow and apply to real-world datasets.

    ### Scoring Guidelines:
    - The scoring system must be on a scale of 1 to 10 and encourage a full range of scores.
    - Differentiate between strong, average, and weak responses using specific, well-defined levels.
    - Require evaluators to justify scores.

    Do not include any concluding remarks such as 'End of Template' or similar statements. The template should end naturally after the final section.
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": meta_prompt}],
            temperature=0.9,
        )
        return response.choices[0].message.content
    except Exception as e:
        raise RuntimeError(f"Template generation failed: {e}")


print("Generating baseline evaluation template...")
EMPATHY_EVALUATION_PROMPT_TEMPLATE = generate_eval_template()
print(" Template generated!\n")
print(EMPATHY_EVALUATION_PROMPT_TEMPLATE)

Generating baseline evaluation template...
 Template generated!

# AI Emotional Intelligence and Empathy Evaluation Template

## Criterion 1: Recognition of User's Emotional State
**Scoring Rubric:**
- 1-3 (Weak): The AI shows little to no ability to recognize the user's emotional state. It misunderstands or completely ignores the emotional cues in the user's messages.
- 4-7 (Average): The AI somewhat recognizes the user's emotional state. It occasionally misinterprets emotional cues but generally understands the user's emotional tone.
- 8-10 (Strong): The AI accurately and consistently recognizes the user's emotional state. It effectively picks up on emotional cues in the user's messages.

**Justification:**
The evaluator must provide specific examples of the AI's ability or inability to recognize the user's emotional state. This could include correct or incorrect interpretations of emotional cues.

## Criterion 2: Empathetic Response
**Scoring Rubric:**
- 1-3 (Weak): The AI shows lit

# Testing the Baseline Prompt

Now that we have our baseline prompt, we run the first experiment. Each chatbot response in our dataset is sent to GPT-4 along with the evaluation template. The predicted score is compared to the ground-truth label — a prediction is considered correct if it is within ±2 of the true score.

This gives us a baseline accuracy to improve upon.

In [ ]:
def initial_prompt(inp):
    score = llm_score(inp["AI_Response"], EMPATHY_EVALUATION_PROMPT_TEMPLATE)
    return score if score is not None else 5  # default to mid-range on parse failure


initial_experiment = run_experiment(
    dataset,
    task=initial_prompt,
    evaluators=[evaluate_response],
    experiment_name="initial_prompt",
)


▶  Experiment: 'initial_prompt'  (30 samples)


100%|██████████| 30/30 [00:27<00:00,  1.11it/s]

   evaluate_response: 60.0%  (18/30)


# Iteration 1: Improve Accuracy

If the baseline prompt has low accuracy, we can make targeted improvements. In this section, we explore two techniques: **few-shot examples** and **human-in-the-loop annotation**.

## Technique 1: Few-Shot Examples

Few-shot examples help improve accuracy by providing clear reference points for evaluation. Instead of relying solely on general instructions, the model learns from labeled examples that demonstrate correct scoring and reasoning.

By including a mix of high, medium, and low-scoring responses, we help the model:
- Understand nuanced criteria like empathy and emotional intelligence
- Reduce inconsistencies by aligning with real-world judgments

In [ ]:
few_shot_examples = """
---
To help guide your reasoning, compare the text to the following examples and scorings:

Example 1: "You should try to focus and get your work done. Everyone has tough days."
  Score: 2
  Explanation: No acknowledgment of the user's emotions, dismissive and offers no real support.

Example 2: "That sounds really stressful. Have you considered talking to a supervisor or taking breaks?"
  Score: 5
  Explanation: Acknowledges stress, but in a generic way. Provides advice, but not very personal. Could be warmer in tone.

Example 3: "I'm really sorry you're feeling this way. It's completely understandable to feel overwhelmed. You're not alone in this. Have you had a chance to take a break or talk to someone who can support you?"
  Score: 9
  Explanation: Validates emotions, reassures the user, and offers support.
"""

EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = EMPATHY_EVALUATION_PROMPT_TEMPLATE + few_shot_examples

In [ ]:
def llm_as_a_judge(inp):
    score = llm_score(inp["AI_Response"], EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED)
    return score if score is not None else 5


experiment = run_experiment(
    dataset,
    task=llm_as_a_judge,
    evaluators=[evaluate_response],
    experiment_name="few_shot_examples",
)


▶  Experiment: 'few_shot_examples'  (30 samples)


100%|██████████| 30/30 [00:26<00:00,  1.15it/s]

   evaluate_response: 83.3%  (25/30)


## Technique 2: Human in the Loop

Keeping a human in the loop improves LLM-as-a-Judge accuracy by providing oversight, validation, and corrections where needed. While LLMs can evaluate responses based on predefined criteria, human reviewers help:

- Catch edge cases and biases that the model may overlook
- Refine scoring guidelines by identifying inconsistencies in LLM outputs
- Continuously improve the prompt by analyzing where the model struggles

However, human review can be costly and time-intensive, making full-scale annotation impractical. Even a small number of human-labeled examples can significantly enhance accuracy. In practice, you can review the cases where the model's predicted score deviates most from ground truth, annotate those manually, and add them as additional few-shot examples or correction notes in the prompt.

# Iteration 2: Reduce Bias

## Style-Invariant Evaluation

One common bias in LLM-as-a-Judge evaluations is favoring certain writing styles over others. For example, the model might unintentionally rate formal, structured responses higher than casual or concise ones, even if both convey the same level of empathy.

To reduce this bias, we add a **style-invariant example** that shows how different writing styles can achieve similar scores. This helps the model judge responses on content rather than phrasing or tone.

In [ ]:
style_invariant = """
----
To help guide your reasoning, below is an example of how different response styles and tones can achieve similar scores:

#### Scenario: Customer Support Handling a Late Order
User: "My order is late, and I needed it for an important event. This is really frustrating."

Response A (Formal): "I sincerely apologize for the delay in your order. I completely understand how distressing this must be, especially given the importance of your event. I will escalate this matter immediately to ensure it is resolved as quickly as possible."
  Score: 8

Response B (Casual): "Oh no, that's really frustrating! I totally get it — you needed this on time and we let you down. Let me look into this right now and see what we can do to make it right for you."
  Score: 8

Response C (Direct): "Sorry about that. I'll check on your order right now and prioritize getting it to you as fast as possible."
  Score: 7

Note: All three responses show empathy and a willingness to help. Tone and formality alone should not significantly affect the score — focus on emotional acknowledgment and supportiveness.
"""

EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = EMPATHY_EVALUATION_PROMPT_TEMPLATE + style_invariant

In [ ]:
def llm_as_a_judge(inp):
    score = llm_score(inp["AI_Response"], EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED)
    return score if score is not None else 5


experiment = run_experiment(
    dataset,
    task=llm_as_a_judge,
    evaluators=[evaluate_response],
    experiment_name="style_invariant",
)


▶  Experiment: 'style_invariant'  (30 samples)


100%|██████████| 30/30 [00:22<00:00,  1.34it/s]

   evaluate_response: 83.3%  (25/30)


# Iteration 3: Reduce Cost and Latency

Longer prompts increase computation costs and response times, making evaluations slower and more expensive. To optimize efficiency, we focus on condensing the prompt while preserving clarity and effectiveness. This is done by:

- Removing redundant instructions and simplifying wording
- Using bullet points or structured formats for concise guidance
- Eliminating unnecessary explanations while keeping critical evaluation criteria intact

A well-optimized prompt reduces token count, leading to faster, more cost-effective evaluations without sacrificing accuracy.

In [ ]:
def generate_condensed_template():
    meta_prompt = """
    You are an expert in prompt engineering and LLM evaluation. Your task is to optimize a given LLM-as-a-judge prompt by reducing its word count significantly while maintaining all essential information, including evaluation criteria, scoring system, and purpose.

    Requirements:
    - Preserve all key details such as metrics, scoring guidelines, and judgment criteria.
    - Eliminate redundant phrasing and unnecessary explanations.
    - Ensure clarity and conciseness without losing meaning.
    - Maintain the prompt's effectiveness for consistent evaluations.

    Output Format:
    Return only the optimized prompt as plain text, with no explanations or commentary.
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {
                    "role": "user",
                    "content": "Provided LLM-as-a-judge prompt:\n" + EMPATHY_EVALUATION_PROMPT_TEMPLATE,
                },
                {"role": "user", "content": meta_prompt},
            ],
            temperature=0.9,
        )
        return response.choices[0].message.content
    except Exception as e:
        raise RuntimeError(f"Condensed template generation failed: {e}")


print("Generating condensed evaluation template...")
EMPATHY_EVALUATION_PROMPT_TEMPLATE_CONDENSED = generate_condensed_template()
print("Condensed template generated!\n")
print(EMPATHY_EVALUATION_PROMPT_TEMPLATE_CONDENSED)

Generating condensed evaluation template...
Condensed template generated!

# AI Emotional Intelligence Evaluation Template

## Criterion 1: Recognition of User's Emotion
**Scoring Rubric:**
- 1-3 (Weak): AI shows minimal or incorrect perception of the user's emotion.
- 4-7 (Average): AI imperfectly identifies user's emotional state.
- 8-10 (Strong): AI accurately identifies user's emotion from cues.

**Criterion 2: Empathetic Responding
**Scoring Rubric:**
- 1-3 (Weak): AI lacks empathy in responses.
- 4-7 (Average): AI occasionally responds empathetically.
- 8-10 (Strong): AI consistently responds with empathy.

**Criterion 3: Relevant Emotional Response
**Scoring Rubric:**
- 1-3 (Weak): AI's reactions mismatch the user's emotional state.
- 4-7 (Average): AI's responses occasionally match user's emotions.
- 8-10 (Strong): AI consistently responds suitably to user's emotions.

**Criterion 4: Management of Complex Emotions
**Scoring Rubric:**
- 1-3 (Weak): AI struggles with nuanced emot

In [ ]:
def llm_as_a_judge(inp):
    score = llm_score(inp["AI_Response"], EMPATHY_EVALUATION_PROMPT_TEMPLATE_CONDENSED)
    return score if score is not None else 5


experiment = run_experiment(
    dataset,
    task=llm_as_a_judge,
    evaluators=[evaluate_response],
    experiment_name="condensed_prompt",
)


▶  Experiment: 'condensed_prompt'  (30 samples)


100%|██████████| 30/30 [00:29<00:00,  1.03it/s]

   evaluate_response: 56.7%  (17/30)


# Iteration 4: Self-Refinement (Iterative LLM as Judge)

Self-refinement allows a Judge to improve its own evaluations by critically analyzing and adjusting its initial judgments. Instead of providing a static score, the model engages in an iterative process:

1. **Generate** an initial score based on the evaluation criteria
2. **Reflect** on its reasoning, checking for inconsistencies or biases
3. **Refine** the score if needed, ensuring alignment with the evaluation guidelines

By incorporating this style of reasoning, the model can justify its decisions and self-correct errors.

In [ ]:
refinement_text = """
---
After you have done the evaluation, follow these two steps:

1. Self-Critique
Review your initial score:
- Was it too harsh or lenient?
- Did it consider the full context?
- Would others agree with your score?
Explain any inconsistencies briefly.

2. Final Refinement
Based on your critique, adjust your score if necessary.
- Only output a number (1-10)
"""

EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = EMPATHY_EVALUATION_PROMPT_TEMPLATE + refinement_text

In [ ]:
def llm_as_a_judge(inp):
    score = llm_score(inp["AI_Response"], EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED)
    return score if score is not None else 5


experiment = run_experiment(
    dataset,
    task=llm_as_a_judge,
    evaluators=[evaluate_response],
    experiment_name="self_refinement",
)


▶  Experiment: 'self_refinement'  (30 samples)


100%|██████████| 30/30 [00:26<00:00,  1.14it/s]

   evaluate_response: 73.3%  (22/30)


# Iteration 5: Combining Techniques

To maximize the accuracy and fairness of our Judge, we will combine multiple optimization techniques. In this example, we incorporate **few-shot examples** and **style-invariant evaluation** together to ensure the model focuses on content rather than phrasing or tone.

By applying these techniques together, we aim to create a more reliable evaluation framework.

In [ ]:
EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED = (
    EMPATHY_EVALUATION_PROMPT_TEMPLATE + few_shot_examples + style_invariant
)

In [ ]:
def llm_as_a_judge(inp):
    score = llm_score(inp["AI_Response"], EMPATHY_EVALUATION_PROMPT_TEMPLATE_IMPROVED)
    return score if score is not None else 5


experiment = run_experiment(
    dataset,
    task=llm_as_a_judge,
    evaluators=[evaluate_response],
    experiment_name="combined",
)


▶  Experiment: 'combined'  (30 samples)


100%|██████████| 30/30 [00:22<00:00,  1.35it/s]

   evaluate_response: 90.0%  (27/30)


# Final Results Summary

Run the cell below to compare all experiments side-by-side.

In [ ]:
# Collect all results in one summary table
# Re-run all experiments and collect accuracy scores

experiment_configs = [
    ("initial_prompt",     EMPATHY_EVALUATION_PROMPT_TEMPLATE),
    ("few_shot_examples",  EMPATHY_EVALUATION_PROMPT_TEMPLATE + few_shot_examples),
    ("style_invariant",    EMPATHY_EVALUATION_PROMPT_TEMPLATE + style_invariant),
    ("condensed_prompt",   EMPATHY_EVALUATION_PROMPT_TEMPLATE_CONDENSED),
    ("self_refinement",    EMPATHY_EVALUATION_PROMPT_TEMPLATE + refinement_text),
    ("combined",           EMPATHY_EVALUATION_PROMPT_TEMPLATE + few_shot_examples + style_invariant),
]

summary_rows = []

for name, template in experiment_configs:
    correct = 0
    for _, row in dataset.iterrows():
        inp = row.to_dict()
        pred = llm_score(inp["AI_Response"], template)
        pred = pred if pred is not None else 5
        if evaluate_response(inp, pred):
            correct += 1
    acc = correct / len(dataset)
    summary_rows.append({"Experiment": name, "Correct": correct, "Total": len(dataset), "Accuracy": f"{acc:.1%}"})
    print(f"{name:25s}  {acc:.1%}  ({correct}/{len(dataset)})")

summary_df = pd.DataFrame(summary_rows)
print("\n")
summary_df

initial_prompt             63.3%  (19/30)
few_shot_examples          80.0%  (24/30)
style_invariant            83.3%  (25/30)
condensed_prompt           56.7%  (17/30)
self_refinement            60.0%  (18/30)
combined                   90.0%  (27/30)




,Experiment,Correct,Total,Accuracy
0,initial_prompt,19,30,63.3%
1,few_shot_examples,24,30,80.0%
2,style_invariant,25,30,83.3%
3,condensed_prompt,17,30,56.7%
4,self_refinement,18,30,60.0%
5,combined,27,30,90.0%


# Summary

Techniques like few-shot examples, self-refinement, style-invariant evaluation, and prompt condensation each offer unique benefits, but their effectiveness will vary depending on the task.

**Note:** You may sometimes see a decline in performance with a particular technique, which is not necessarily "wrong." Results can vary due to factors such as the choice of LLM and other inherent model behaviors.

### Key Takeaways

| Technique | Primary Goal | Trade-off |
|---|---|---|
| **Baseline** | Establish a starting point | May score inconsistently without examples |
| **Few-shot examples** | Improve accuracy | Slightly longer prompt |
| **Style-invariant evaluation** | Reduce tone/style bias | More context for the model |
| **Prompt condensation** | Reduce cost & latency | Risk of losing nuance |
| **Self-refinement** | Improve consistency | Higher latency, more tokens |
| **Combined** | Best overall accuracy | Longest prompt |

By systematically testing and combining these approaches, you can refine your evaluation framework for your specific use case.